# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 4: Preprocessing & Train/Test Split

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('churn_cleaned.csv')
print(f'Cleaned data loaded: {df.shape[0]} rows × {df.shape[1]} columns ✓')

### 4.1 — Separate Features and Target

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nChurn distribution in y:')
print(y.value_counts())
print(f'\nClass balance: {y.mean()*100:.1f}% churn')

### 4.2 — Encode Binary Categorical Columns (Yes/No → 1/0)

In [ ]:
# Columns that are simple Yes/No
binary_cols = [
    'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'PaperlessBilling'
]

for col in binary_cols:
    X[col] = X[col].map({'Yes': 1, 'No': 0})

# Gender: Female → 1, Male → 0
X['gender'] = X['gender'].map({'Female': 1, 'Male': 0})

print('Binary columns encoded ✓')
print(X[binary_cols + ['gender']].head(3))

### 4.3 — One-Hot Encode Multi-Class Categorical Columns

In [ ]:
# These columns have 3+ categories — use one-hot encoding
multi_cols = ['InternetService', 'Contract', 'PaymentMethod']

print('Before encoding:')
for col in multi_cols:
    print(f'  {col}: {df[col].unique()}')

X = pd.get_dummies(X, columns=multi_cols, drop_first=False)

print(f'\nAfter one-hot encoding:')
print(f'  Feature count: {X.shape[1]} columns')
print(f'  New columns added: {[c for c in X.columns if any(m in c for m in multi_cols)]}')

### 4.4 — Confirm All Columns Are Numeric

In [ ]:
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()

if len(non_numeric) == 0:
    print('All features are numeric ✓')
else:
    print(f'WARNING — Non-numeric columns still present: {non_numeric}')

print(f'\nFinal feature matrix shape: {X.shape}')
print(f'Feature names:\n{list(X.columns)}')

### 4.5 — Train / Test Split (70% / 30% Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y        # preserves churn ratio in both splits
)

print(f'Training set : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set     : {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nChurn rate — Train : {y_train.mean()*100:.1f}%')
print(f'Churn rate — Test  : {y_test.mean()*100:.1f}%')
print('\nStratification confirmed — churn ratio preserved in both sets ✓')

### 4.6 — Scale Numeric Features

In [ ]:
# Scale only the 3 continuous numeric columns
# Binary/one-hot encoded columns do NOT need scaling
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()

# Fit on TRAIN only — then transform both
# (never fit on test — that would cause data leakage)
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print('StandardScaler applied to:', scale_cols)
print('Fit on training set only — no data leakage ✓')
print(f'\nSample scaled values (train):')
print(X_train[scale_cols].describe().round(3))

### 4.7 — Visualise Train/Test Split Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, y_split, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    counts = y_split.value_counts()
    ax.bar(['No Churn', 'Churn'], counts.values,
           color=['#2E86AB', '#E74C3C'], edgecolor='white', width=0.5)
    for i, (val, cnt) in enumerate(zip(['No Churn', 'Churn'], counts.values)):
        ax.text(i, cnt + 10, f'{cnt}\n({cnt/len(y_split)*100:.1f}%)',
                ha='center', fontweight='bold')
    ax.set_title(f'{title} (n={len(y_split):,})', fontweight='bold', fontsize=12)
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts.values) * 1.15)

plt.suptitle('Class Distribution — Train vs Test Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_10_train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 4.8 — Save Preprocessed Data

In [ ]:
# Save all four splits so Steps 5–7 can load them directly
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',  index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',  index=False)

print('Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv ✓')

print('\n' + '='*55)
print('        PREPROCESSING SUMMARY')
print('='*55)
print(f'  ✓ Binary columns encoded  (Yes/No → 1/0)')
print(f'  ✓ Multi-class columns one-hot encoded')
print(f'  ✓ All features confirmed numeric')
print(f'  ✓ 70/30 stratified train/test split')
print(f'  ✓ StandardScaler applied (fit on train only)')
print(f'  ✓ Training rows : {X_train.shape[0]:,}')
print(f'  ✓ Test rows     : {X_test.shape[0]:,}')
print(f'  ✓ Final features: {X_train.shape[1]}')
print('='*55)
print('Step 4 Complete ✓  →  Proceed to Step 5: Model Training')